# 04 - MAGMA gene-level association

Two MAGMA v1.10 steps: annotate variants to genes (NCBI38 gene coordinates), then the
gene-level analysis using the 1000 Genomes European reference panel for LD.

The binary and the reference panel are not in this repository - see `tools/README.md`.
The gene analysis takes roughly 20 minutes; both steps skip themselves if their output already
exists, so re-running this notebook is cheap. Delete `interim/magma/` to force a fresh run.

Output: `results/tables/04_magma_genes.tsv` (gene-level Z and p, with symbols).

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
from adpipe import *

import pandas as pd

SNP_LOC = INTERIM / "magma_input.txt"
ANNOT = MAGMA_DIR / "magma_annotated"
GENES = MAGMA_DIR / "magma_genes"

assert SNP_LOC.exists(), "run notebook 03 first - interim/magma_input.txt is missing"
print("MAGMA tool dir:", MAGMA_TOOL, "| exists:", MAGMA_TOOL.exists())

MAGMA tool dir: C:\Users\Admin\Desktop\azihmer\tools\magma | exists: True


In [2]:
# Step 1 - annotate variants to genes.  Paths are relative to tools/magma (MAGMA's cwd).
if (ANNOT.with_suffix(".genes.annot.txt")).exists():
    print("annotation already present, skipping:", ANNOT.with_suffix('.genes.annot.txt').name)
else:
    import os
    rel = lambda p: os.path.relpath(p, MAGMA_TOOL)
    run_magma(["--annotate",
               "--snp-loc", rel(SNP_LOC),
               "--gene-loc", "NCBI38.gene.loc",
               "--out", rel(ANNOT)])

annotation already present, skipping: magma_annotated.genes.annot.txt


In [3]:
# Step 2 - gene analysis (SNP-wise mean model).  ~20 minutes.
if GENES.with_suffix(".genes.out.txt").exists():
    print("gene analysis already present, skipping:", GENES.with_suffix('.genes.out.txt').name)
else:
    import os
    rel = lambda p: os.path.relpath(p, MAGMA_TOOL)
    run_magma(["--bfile", "g1000_eur",
               "--pval", rel(SNP_LOC), "ncol=N",
               "--gene-annot", rel(ANNOT.with_suffix(".genes.annot.txt")),
               "--out", rel(GENES)])

$ C:\Users\Admin\Desktop\azihmer\tools\magma\magma.exe --bfile g1000_eur --pval ..\..\interim\magma_input.txt ncol=N --gene-annot ..\..\interim\magma\magma_annotated.genes.annot.txt --out ..\..\interim\magma\magma_genes


 (90%)     
	processed genes: 16780 (90.1%)     
	processed genes: 16798 (90.2%)     
	processed genes: 16818 (90.3%)     
	processed genes: 16849 (90.5%)     
	processed genes: 16868 (90.6%)     
	processed genes: 16890 (90.7%)     
	processed genes: 16922 (90.9%)     
	processed genes: 16938 (90.9%)     
	processed genes: 16956 (91%)     
	processed genes: 16973 (91.1%)     
	processed genes: 16977 (91.1%)     
	processed genes: 16981 (91.2%)     
	processed genes: 16986 (91.2%)     
	processed genes: 16988 (91.2%)     
	processed genes: 16995 (91.2%)     
	processed genes: 17009 (91.3%)     
	processed genes: 17029 (91.4%)     
	processed genes: 17047 (91.5%)     
	processed genes: 17063 (91.6%)     
	processed genes: 17074 (91.7%)     
	processed genes: 17089 (91.7%)     
	processed genes: 17110 (91.9%)     
	processed genes: 17136 (92%)     
	processed genes: 17157 (92.1%)     
	processed genes: 17171 (92.2%)     
	processed genes: 17188 (92.3%)     
	processed genes: 17209 (92.4%

In [4]:
genes = pd.read_csv(GENES.with_suffix(".genes.out.txt"), sep=r"\s+")
gene_loc = pd.read_csv(MAGMA_TOOL / "NCBI38.gene.loc", sep="\t", header=None,
                       names=["GENE", "CHR", "START", "STOP", "STRAND", "SYMBOL"])
genes = genes.merge(gene_loc[["GENE", "SYMBOL"]], on="GENE", how="left")

print(f"genes tested: {len(genes):,} | with a symbol: {genes['SYMBOL'].notna().sum():,}")
print(f"Bonferroni threshold: {0.05 / len(genes):.3e}")
print(f"genes passing it    : {(genes['P'] < 0.05 / len(genes)).sum()}")

save_table(genes, "04_magma_genes", index=False)
genes.sort_values("P").head(20)[["GENE", "SYMBOL", "CHR", "NSNPS", "ZSTAT", "P"]]

genes tested: 18,626 | with a symbol: 18,626
Bonferroni threshold: 2.684e-06
genes passing it    : 147


  table -> results/tables/04_magma_genes.tsv  (18,626 rows x 10 cols, 1155 KB)


,GENE,SYMBOL,CHR,NSNPS,ZSTAT,P
16983,341,APOC1,19,1,11.0110,1.700000e-28
16985,344,APOC2,19,13,8.8056,6.505200e-19
3163,3635,INPP5D,2,518,8.5081,8.841600e-18
16977,602,BCL3,19,20,8.0194,5.315000e-16
10548,64231,MS4A6A,11,20,7.8970,1.428100e-15
16979,4059,BCAM,19,33,7.8533,2.025800e-15
16995,90332,EXOC3L2,19,96,7.7710,3.894900e-15
10549,643680,MS4A4E,11,109,7.6669,8.807400e-15
2653,274,BIN1,2,222,7.6477,1.023200e-14
16978,23624,CBLC,19,51,7.6384,1.099800e-14
